In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import scipy
from snf.farfield import load_far_field, ff_to_T, T_to_R
from snf.fields import load_field, rotate_by_angles, s21_reciprocity
from snf.swe import psi, j_index
from snf.sampling import *
from snf.solve import lstsq, omp

DATA = '../data/'
os.makedirs(DATA + 'cs', exist_ok=True)

# Define Setup

In [ ]:
freq = 15 * 10**9
k = 2 * np.pi * freq / scipy.constants.c
R_PROBE = 0.4 # Distance from AUT origin to probe far field origin (m)
r0 = 0.095 # AUT minimum sphere radius (m)

N_AUT = 40
N_PROBE = 12
mus = np.arange(-3, 3 + 1) # probe mu used when solving
mus_gen = np.arange(-N_PROBE, N_PROBE + 1) # probe mu used to generate model measurements

probe_wp = 1
aut_wp = 1

# Ground Truth SWEC from Far Field

In [ ]:
T_probe = ff_to_T(*load_far_field(DATA + f'FF_fields/FF_probe_wp{probe_wp}.csv'), N_PROBE)
R_probe = T_to_R(T_probe)
T_aut = ff_to_T(*load_far_field(DATA + f'FF_fields/FF_AUT_wp{aut_wp}.csv'), N_AUT)

err_db = lambda T: 20 * np.log10(np.linalg.norm(T - T_aut) / np.linalg.norm(T_aut)) # relative coefficient error (dB)

# Sparsity of T

In [ ]:
# |T_smn| on the (n, m) plane for each s: the valid region |m| <= n is the V

T_db = 20 * np.log10(np.abs(T_aut) / np.abs(T_aut).max())
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for s, ax in zip((1, 2), axes):
    img = np.full((2 * N_AUT + 1, N_AUT), np.nan)
    for n in range(1, N_AUT + 1):
        ms = np.arange(-n, n + 1)
        img[ms + N_AUT, n - 1] = T_db[j_index(s, ms, n)]
    im = ax.imshow(img, origin='lower', aspect='auto', extent=(0.5, N_AUT + 0.5, -N_AUT - 0.5, N_AUT + 0.5), vmin=-100, vmax=0, cmap='viridis')
    ax.set(xlabel='n', title=f's = {s}')
axes[0].set_ylabel('m')
fig.colorbar(im, ax=axes, label='|T| (dB rel. max)')

In [ ]:
# |T_j| against Hansen index j, and sorted coefficient decay

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(np.arange(1, len(T_aut) + 1), T_db, 'k.', ms=2)
ax1.set(xlabel='j', ylabel='|T_j| (dB rel. max)', title='Coefficients by Hansen index', ylim=(-160, 5))
p = np.sort(np.abs(T_aut)**2)[::-1]
ax2.semilogx(np.arange(1, len(p) + 1), 10 * np.log10(p / p[0]), 'k-')
ax2.set(xlabel='Number of coefficients (sorted)', ylabel='|T|^2 (dB rel. max)', title='Sorted coefficient decay', ylim=(-160, 5))
for ax in (ax1, ax2):
    ax.grid(True)

c = np.cumsum(p) / p.sum()
print(f'Unknowns: {len(T_aut)}')
for f in (0.99, 0.999, 0.9999, 0.99999):
    print(f'Coefficients holding {f * 100:g}% of power: {np.searchsorted(c, f) + 1}')
for dB in (-40, -60, -80):
    print(f'Coefficients above {dB} dB of max: {np.sum(p > p[0] * 10**(dB / 10))}')
m1 = sum(np.sum(np.abs(T_aut[j_index(s, m, np.arange(1, N_AUT + 1))])**2) for s in (1, 2) for m in (-1, 1))
print(f'Power fraction in m = +/-1: {m1 / p.sum():.5f}')

# Helper Functions

In [ ]:
# Full grid reciprocity measurements (from compare_swec)
theta_full, phi_full = full_sampling(N_AUT)
w_full = np.load(DATA + f'w_full_probe{probe_wp}_aut{aut_wp}.npy')

fields = None
def reciprocity(chi, theta, phi): # S21 by reciprocity at each Euler angle (radians, HFSS convention)
    global fields
    if fields is None: # load fields only when needed
        xyz, Ea = load_field(DATA + f'SNF_fields_ports/probe/wp{probe_wp}/E.csv')
        _, Ha = load_field(DATA + f'SNF_fields_ports/probe/wp{probe_wp}/H.csv')
        _, Eb = load_field(DATA + f'SNF_fields_ports/AUT/wp{aut_wp}/E.csv')
        _, Hb = load_field(DATA + f'SNF_fields_ports/AUT/wp{aut_wp}/H.csv')
        fields = (xyz, Ea, Ha, Eb, Hb)
    xyz, Ea, Ha, Eb, Hb = fields
    w = np.zeros(len(theta), dtype=complex)
    for i, (th, ph, ch) in enumerate(zip(theta, phi, chi)):
        Ea_rot = rotate_by_angles(xyz, Ea, th, ph, ch, degrees=False)
        Ha_rot = rotate_by_angles(xyz, Ha, th, ph, ch, degrees=False)
        w[i] = s21_reciprocity(xyz, Ea_rot, Ha_rot, Eb, Hb)
    return w

def get_measurements(scheme, num_points, seed): # Sample positions for a scheme, add chi = 0, 90 deg, and get reciprocity w (cached in data/cs)
    np.random.seed(seed)
    if scheme == 'equiangular_bernoulli':
        theta, phi, idx = equiangular_bernoulli(num_points, N_AUT)
        chi, theta, phi = add_chi(theta, phi)
        return chi, theta, phi, w_full[np.r_[idx, idx + len(theta_full)]]
    path = DATA + f'cs/{scheme}_P{num_points}_seed{seed}.npz'
    if os.path.exists(path):
        d = np.load(path)
        return d['chi'], d['theta'], d['phi'], d['w']
    if scheme == 'spiral_random_phi':
        theta, phi = spiral_random_phi(num_points)
    elif scheme == 'constant_velocity':
        theta, phi = constant_velocity(num_points)
    elif scheme == 'randomized_constant_velocity':
        theta, phi = randomized_constant_velocity(num_points)
    elif scheme == 'uniform_random':
        theta, phi = uniform_random(num_points)
    chi, theta, phi = add_chi(theta, phi)
    w = reciprocity(chi, theta, phi)
    np.savez(path, chi=chi, theta=theta, phi=phi, w=w)
    return chi, theta, phi, w

def model_measurements(chi, theta, phi): # w = Psi T_aut with all probe mu, in HFSS convention like reciprocity
    Psi_gen = psi(chi, theta, phi, R_probe, N_AUT, k * R_PROBE, N_PROBE, mus_gen)
    return (Psi_gen @ T_aut).conj()

def recover(chi, theta, phi, w, tol): # LS and OMP recovery of T from HFSS-convention w
    Psi, sw = psi(chi, theta, phi, R_probe, N_AUT, k * R_PROBE, N_PROBE, mus, weighted=True)
    w_p = w.conj() * sw # Hansen convention, weighted
    return Psi, lstsq(Psi, w_p), omp(Psi, w_p, tol, max_k=min(len(w) // 2, 600))

In [ ]:
# OMP stopping tolerance: relative residual of full-grid LS (reciprocity / model mismatch floor)
chi, theta, phi = add_chi(theta_full, phi_full)
Psi, sw = psi(chi, theta, phi, R_probe, N_AUT, k * R_PROBE, N_PROBE, mus, weighted=True)
w_p = w_full.conj() * sw
T_full = lstsq(Psi, w_p)
tol = 1.1 * np.linalg.norm(w_p - Psi @ T_full) / np.linalg.norm(w_p)
full_err = err_db(T_full)
print(f'Full grid LS error: {full_err:.1f} dB, OMP tolerance: {tol:.2e}')

# Sample and Recover

In [ ]:
num_points = 500
seed = 0

scheme = 'equiangular_bernoulli'
# scheme = 'spiral_random_phi'
# scheme = 'constant_velocity'
# scheme = 'randomized_constant_velocity'
# scheme = 'uniform_random'

chi, theta, phi, w = get_measurements(scheme, num_points, seed)
# w = model_measurements(chi, theta, phi) # model-generated instead of reciprocity

th_pos, ph_pos = sphere_position(theta, phi)
plt.scatter(np.degrees(th_pos), np.degrees(ph_pos), s=4)
plt.xlabel('Theta (deg)')
plt.ylabel('Phi (deg)')
plt.title(f'{scheme}, {num_points} positions')

In [ ]:
Psi, T_ls, T_omp = recover(chi, theta, phi, w, tol)
print(f'M = {len(w)}, coherence {coherence(Psi):.3f}')
print(f'LS error: {err_db(T_ls):.1f} dB, OMP error: {err_db(T_omp):.1f} dB ({np.count_nonzero(T_omp)} coefficients)')

# Power per n: truth, recovered, and error (all relative to total true power)
power_n = lambda T: np.array([np.sum(np.abs(T[j_index(np.array([1, 2])[:, None], np.arange(-n, n + 1)[None], n)])**2) for n in range(1, N_AUT + 1)])
ns, P_tot = np.arange(1, N_AUT + 1), np.sum(np.abs(T_aut)**2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (T_rec, name) in zip(axes, ((T_ls, 'LS'), (T_omp, 'OMP'))):
    ax.plot(ns, 10 * np.log10(power_n(T_aut) / P_tot), label='True AUT T')
    ax.plot(ns, 10 * np.log10(power_n(T_rec) / P_tot + 1e-30), '--', label='Recovered AUT T')
    ax.plot(ns, 10 * np.log10(power_n(T_rec - T_aut) / P_tot), label='Error')
    ax.axvline(k * r0, color='gray', ls='--', label='k r0')
    ax.set(xlabel='n', title=f'{scheme}, M = {len(w)}: {name}', ylim=(-150, 5))
    ax.grid(True); ax.legend()
axes[0].set_ylabel('Power relative to total (dB)')

# Sweep Number of Measurements

In [ ]:
# Sweep positions and seeds for one scheme; save results to data/cs/results_{scheme}_{source}.npz

scheme = 'equiangular_bernoulli'
# scheme = 'spiral_random_phi'
# scheme = 'constant_velocity'
# scheme = 'randomized_constant_velocity'
# scheme = 'uniform_random'

source = 'reciprocity'
# source = 'model'

Ps = [100, 200, 300, 500, 750, 1000, 1500, 2000] + ([len(theta_full)] if scheme == 'equiangular_bernoulli' else [])
n_seeds = {'equiangular_bernoulli': 10, 'constant_velocity': 1, 'randomized_constant_velocity': 5}.get(scheme, 2)

res = {key: [] for key in ('P', 'seed', 'M', 'err_ls', 'err_omp', 'k_omp', 'coherence', 'T_ls', 'T_omp')}
for P in Ps:
    for seed in range(1 if P == len(theta_full) else n_seeds):
        chi, theta, phi, w = get_measurements(scheme, P, seed)
        if source == 'model':
            w = model_measurements(chi, theta, phi)
        Psi, T_ls, T_omp = recover(chi, theta, phi, w, tol)
        for key, val in zip(res, (P, seed, len(w), err_db(T_ls), err_db(T_omp), np.count_nonzero(T_omp),
                                  coherence(Psi) if source == 'reciprocity' else np.nan, T_ls, T_omp)):
            res[key].append(val)
        print(f'{scheme} {source} P={P} seed={seed}: LS {res["err_ls"][-1]:.1f} dB, OMP {res["err_omp"][-1]:.1f} dB')
np.savez(DATA + f'cs/results_{scheme}_{source}.npz', **{key: np.array(val) for key, val in res.items()})

In [ ]:
# Noise robustness: add complex Gaussian noise to reciprocity w and re-solve with OMP

scheme = 'equiangular_bernoulli'
# scheme = 'randomized_constant_velocity'

SNRs = [60, 40, 20] # dB, relative to rms(w)
Ps = [100, 200, 300, 500, 750, 1000, 1500, 2000]
n_seeds = 3

for snr in SNRs:
    res = {key: [] for key in ('P', 'seed', 'M', 'err_omp', 'k_omp')}
    for P in Ps:
        for seed in range(n_seeds):
            chi, theta, phi, w = get_measurements(scheme, P, seed)
            rng = np.random.default_rng(seed)
            sigma = np.sqrt(np.mean(np.abs(w)**2)) * 10**(-snr / 20)
            w_n = w + sigma / np.sqrt(2) * (rng.normal(size=w.size) + 1j * rng.normal(size=w.size))
            _, _, T_omp = recover(chi, theta, phi, w_n, 1.1 * np.hypot(tol, 10**(-snr / 20)))
            for key, val in zip(res, (P, seed, len(w), err_db(T_omp), np.count_nonzero(T_omp))):
                res[key].append(val)
        print(f'{scheme} SNR {snr} dB done')
    np.savez(DATA + f'cs/noise_{scheme}_snr{snr}.npz', **{key: np.array(val) for key, val in res.items()})

# Results

In [ ]:
# Load every saved sweep

results = {os.path.basename(f)[8:-4]: dict(np.load(f)) for f in sorted(glob.glob(DATA + 'cs/results_*.npz'))}
print(list(results))

def stats(r, key): # median, min, max over seeds for each M
    Ms = np.unique(r['M'])
    vals = [r[key][r['M'] == M] for M in Ms]
    return Ms, np.array([np.median(v) for v in vals]), np.array([v.min() for v in vals]), np.array([v.max() for v in vals])

In [ ]:
# Error vs number of measurements for each scheme (reciprocity measurements)

schemes = [name[:-12] for name in results if name.endswith('_reciprocity')]
fig, axes = plt.subplots(int(np.ceil(len(schemes) / 2)), 2, figsize=(12, 4.5 * np.ceil(len(schemes) / 2)), squeeze=False)
for ax, scheme in zip(axes.flat, schemes):
    r = results[scheme + '_reciprocity']
    for key, color, name in (('err_ls', 'C0', 'LS'), ('err_omp', 'C1', 'OMP')):
        Ms, med, lo, hi = stats(r, key)
        ax.plot(Ms, med, 'o-', color=color, label=name)
        ax.fill_between(Ms, lo, hi, color=color, alpha=0.2)
    ax.axhline(full_err, color='k', ls=':', label='Full grid LS')
    ax.axvline(len(T_aut), color='gray', ls='--', label='Unknowns')
    ax.set(xscale='log', xlabel='Measurements M', ylabel='Relative error (dB)', title=scheme, ylim=(-70, 10))
    ax.grid(True, which='both'); ax.legend()
for ax in axes.flat[len(schemes):]:
    ax.axis('off')
plt.tight_layout()

In [ ]:
# OMP for all schemes, and reciprocity vs model-generated measurements

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for i, scheme in enumerate(schemes):
    Ms, med, lo, hi = stats(results[scheme + '_reciprocity'], 'err_omp')
    ax1.plot(Ms, med, 'o-', color=f'C{i}', label=scheme)
    ax1.fill_between(Ms, lo, hi, color=f'C{i}', alpha=0.15)
    ax2.plot(Ms, med, 'o-', color=f'C{i}', label=f'{scheme} (reciprocity)')
    if scheme + '_model' in results:
        Ms, med, _, _ = stats(results[scheme + '_model'], 'err_omp')
        ax2.plot(Ms, med, 'x--', color=f'C{i}', label=f'{scheme} (model)')
for ax, title in ((ax1, 'OMP, reciprocity measurements'), (ax2, 'OMP: reciprocity vs model-generated')):
    ax.axhline(full_err, color='k', ls=':')
    ax.axvline(len(T_aut), color='gray', ls='--')
    ax.set(xscale='log', xlabel='Measurements M', title=title, ylim=(-70, 10))
    ax.grid(True, which='both'); ax.legend(fontsize=7)
ax1.set_ylabel('Relative error (dB)')
plt.tight_layout()

In [ ]:
# Coherence vs number of measurements

plt.figure(figsize=(7, 4.5))
for i, scheme in enumerate(schemes):
    Ms, med, lo, hi = stats(results[scheme + '_reciprocity'], 'coherence')
    plt.plot(Ms, med, 'o-', color=f'C{i}', label=scheme)
    plt.fill_between(Ms, lo, hi, color=f'C{i}', alpha=0.15)
plt.xscale('log')
plt.xlabel('Measurements M')
plt.ylabel('Coherence of weighted Psi')
plt.grid(True, which='both'); plt.legend(fontsize=7)

In [ ]:
# Noise robustness

noise = {os.path.basename(f)[6:-4]: dict(np.load(f)) for f in sorted(glob.glob(DATA + 'cs/noise_*.npz'))}
noise_schemes = sorted({name.rsplit('_snr', 1)[0] for name in noise})
fig, axes = plt.subplots(1, max(len(noise_schemes), 1), figsize=(6 * max(len(noise_schemes), 1), 4.5), squeeze=False)
for ax, scheme in zip(axes.flat, noise_schemes):
    Ms, med, _, _ = stats(results[scheme + '_reciprocity'], 'err_omp')
    ax.plot(Ms, med, 'ko-', label='No added noise')
    for i, name in enumerate(n for n in noise if n.startswith(scheme)):
        Ms, med, lo, hi = stats(noise[name], 'err_omp')
        ax.plot(Ms, med, 'o-', color=f'C{i}', label=f'SNR {name.rsplit("snr", 1)[1]} dB')
        ax.fill_between(Ms, lo, hi, color=f'C{i}', alpha=0.15)
    ax.axvline(len(T_aut), color='gray', ls='--')
    ax.set(xscale='log', xlabel='Measurements M', ylabel='Relative error (dB)', title=f'{scheme}: OMP with noise', ylim=(-70, 10))
    ax.grid(True, which='both'); ax.legend()
plt.tight_layout()